In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. .env 파일 로드
load_dotenv('../project_db/geo.env')

# 2. 환경변수 읽기
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")  # 기본값 3306
DB_NAME = os.getenv("DB_NAME")

# 3. SQLAlchemy 엔진 생성 (MySQL / MariaDB 기준)
# 만약 PostgreSQL이라면 postgresql:// 로 변경해주세요!
connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

print(f"DB 연결 설정 완료: {DB_NAME}@{DB_HOST}")

DB 연결 설정 완료: geo_db@localhost


In [2]:
# 1. 브랜드명이 '소버먼트'인 raw_data 조회
idylic_raw_df = pd.read_sql(
    "SELECT * FROM raw_data_table WHERE brand_name = '아이딜릭마켓'", 
    con=engine
)

print(f"✅ '아이딜릭마켓' 브랜드 상품 개수: {len(idylic_raw_df)}개")

✅ '아이딜릭마켓' 브랜드 상품 개수: 590개


In [3]:
if not idylic_raw_df.empty:
    target_page_id = idylic_raw_df.iloc[0]['page_id']
    target_product_name = idylic_raw_df.iloc[0]['product_name']
    
    print(f"🎯 테스트 대상 page_id: {target_page_id} (상품명: {target_product_name})")

    # 2. 해당 page_id의 단일 raw_data 및 image_data 추출
    raw_df = idylic_raw_df[idylic_raw_df['page_id'] == target_page_id]
    img_df = pd.read_sql(
        f"SELECT * FROM image_data_table WHERE page_id = {target_page_id} ORDER BY image_sequence ASC", 
        con=engine
    )

    # 3. GEOScorer 입력 형태 전처리
    body_text = raw_df.iloc[0]['text_contents'] if pd.notna(raw_df.iloc[0]['text_contents']) else ""
    json_ld_str = raw_df.iloc[0]['json_ld_contents'] if pd.notna(raw_df.iloc[0]['json_ld_contents']) else ""

    # 이미지 텍스트 결합 (raw_df에서 가져옴)
    raw_image_text = raw_df.iloc[0]['image_text'] if pd.notna(raw_df.iloc[0]['image_text']) else ""
    image_text_combined = str(raw_image_text).strip()
    
    # 해당 상품에 추출된 이미지 텍스트가 있는지 여부
    has_text_in_product = bool(image_text_combined)

    # 이미지 Alt 속성 리스트 구성
    image_list_for_eval = []
    for _, row in img_df.iterrows():
        alt_val = row['alt_contents'] if row['has_alt'] == 1 and pd.notna(row['alt_contents']) else None
        
        # ✅ row['image_text'] 대신, raw_df 기준 text 포함 여부(has_text_in_product) 적용
        image_list_for_eval.append({
            'alt': alt_val,
            'is_text_image': has_text_in_product
        })

    print("\n=== [아이딜릭마켓 전처리 결과] ===")
    print(f"1. 본문 텍스트 길이: {len(body_text)}자")
    print(f"2. 추출된 이미지 텍스트 길이: {len(image_text_combined)}자")
    print(f"3. JSON-LD 존재 여부: {'존재함' if json_ld_str else '없음'}")
    print(f"4. 평가 대상 이미지 개수: {len(image_list_for_eval)}개")

else:
    print("⚠️ DB에 '아이딜릭마켓' 브랜드 데이터가 존재하지 않습니다. DB의 brand_name 표기를 확인해 주세요.")

🎯 테스트 대상 page_id: 3077 (상품명: 🍀쓰리피스set #살안타템 #레이어드 뷔스티에 블라우스 가디건 세트(2col) 모네블라우스세트)

=== [아이딜릭마켓 전처리 결과] ===
1. 본문 텍스트 길이: 6자
2. 추출된 이미지 텍스트 길이: 1517자
3. JSON-LD 존재 여부: 존재함
4. 평가 대상 이미지 개수: 7개


In [4]:
import pandas as pd
from pathlib import Path
from premodel2_ver210 import GEOScorer

# 1. GEOScorer 객체 생성
scorer = GEOScorer()

# 2. gemini_collected_data.csv에서 질문 데이터 추출
csv_path = Path("gemini_collected_data.csv")

try:
    df_queries = pd.read_csv(csv_path)
except UnicodeDecodeError:
    df_queries = pd.read_csv(csv_path, encoding="cp949")

# 실제 CSV 컬럼명 지정
category_col = "카테고리"
query_col = "답변"

# 카테고리가 '블라우스'인 쿼리 추출
filtered_df = df_queries[df_queries[category_col].astype(str).str.strip() == "블라우스"]

test_user_queries = (
    filtered_df[query_col]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

print(f"🎯 카테고리 '블라우스' 질문 로드 완료: 총 {len(test_user_queries)}개")

# 3. 모델 평가 실행 (전처리 완료된 데이터 사용)
evaluation_result = scorer.evaluate_page(
    body_text=body_text,
    image_text=image_text_combined,
    user_queries=test_user_queries,
    json_ld_str=json_ld_str,
    image_list=image_list_for_eval
)

# 4. 종합 평가 결과 및 세부 점수 출력
print("==================================================")
print(f"🏆 [page_id: {target_page_id}] 최종 GEO 점수: {evaluation_result.formatted_score}")
print("==================================================")
print(f"1. 본문 텍스트 비율 점수 (text_ratio): {evaluation_result.text_ratio_score}")
print(f"2. 하이브리드 검색 점수 (hybrid_search): {evaluation_result.hybrid_search.final_score}")
print(f"3. 키워드 스터핑 점수 (keyword_stuffing): {evaluation_result.keyword_stuffing.final_score} (어뷰징 감지: {evaluation_result.keyword_stuffing.is_stuffing})")
print(f"4. JSON-LD 구조화 점수 (json_ld): {evaluation_result.json_ld.final_score} (유효성: {evaluation_result.json_ld.is_valid})")
print(f"5. 이미지 Alt 점수 (image_alt): {evaluation_result.image_alt.avg_score} (유효성: {evaluation_result.image_alt.is_valid})")
print("==================================================")

c:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
from dataclasses import asdict

# 1. py 파일의 클래스/함수로 계산 실행
# (evaluator는 사용 중이신 인스턴스 변수명, json_ld_data는 DB에서 가져온 데이터)
result = scorer.calculate_json_ld_score(json_ld_str)

# 2. 딕셔너리로 변환해서 출력
asdict(result)


================ [JSON-LD 파싱 완료 내용] ================
{
  "@context": "https://schema.org",
  "@type": "Product",
  "name": "🍀쓰리피스set #살안타템 #레이어드 뷔스티에 블라우스 가디건 세트(2col) 모네블라우스세트",
  "image": [
    "https://idyllic-market.kr/web/product/big/202607/a898ffb7693ed3298832cb988a5ef4ee.webp",
    "https://idyllic-market.kr/web/product/extra/big/202607/37b7acc5eeae92a8d2ee17f3038cc0a8.jpg",
    "https://idyllic-market.kr/web/product/extra/big/202607/505d2fd5211f74912a53c3b999677af2.jpg",
    "https://idyllic-market.kr/web/product/extra/big/202607/48ab86c5b9d861a1f58b3a6230270e3c.jpg"
  ],
  "description": "",
  "brand": {
    "@type": "Brand",
    "name": "아이딜릭"
  },
  "offers": [
    {
      "name": "🍀쓰리피스set #살안타템 #레이어드 뷔스티에 블라우스 가디건 세트(2col) 모네블라우스세트 핑크-F",
      "price": 32900,
      "priceCurrency": "KRW",
      "availability": "InStock",
      "url": "https://idyllic-market.kr/product/🍀쓰리피스set-살안타템-레이어드-뷔스티에-블라우스-가디건-세트2col-모네블라우스세트/3031/?item_code=P0000EMP000A",
      "image": "https://

{'final_score': 0.175,
 'raw_score': 0.175,
 'is_valid': True,
 'parsing_score': 0.7,
 'density_score': 0.0,
 'clothing_score': 0.0,
 'trust_score': 0.0,
 'present_attrs_count': 0,
 'trust_count': 0}

In [ ]:
# 호출부 예시
print("전달할 JSON-LD 데이터:", repr(json_ld_str)) # 데이터가 실제로 들어있는지 확인

전달할 JSON-LD 데이터: '[{"@context": "https://schema.org", "@type": "Product", "name": "🍀쓰리피스set #살안타템 #레이어드 뷔스티에 블라우스 가디건 세트(2col) 모네블라우스세트", "image": ["https://idyllic-market.kr/web/product/big/202607/a898ffb7693ed3298832cb988a5ef4ee.webp", "https://idyllic-market.kr/web/product/extra/big/202607/37b7acc5eeae92a8d2ee17f3038cc0a8.jpg", "https://idyllic-market.kr/web/product/extra/big/202607/505d2fd5211f74912a53c3b999677af2.jpg", "https://idyllic-market.kr/web/product/extra/big/202607/48ab86c5b9d861a1f58b3a6230270e3c.jpg"], "description": "", "brand": {"@type": "Brand", "name": "아이딜릭"}, "offers": [{"name": "🍀쓰리피스set #살안타템 #레이어드 뷔스티에 블라우스 가디건 세트(2col) 모네블라우스세트 핑크-F", "price": 32900, "priceCurrency": "KRW", "availability": "InStock", "url": "https://idyllic-market.kr/product/🍀쓰리피스set-살안타템-레이어드-뷔스티에-블라우스-가디건-세트2col-모네블라우스세트/3031/?item_code=P0000EMP000A", "image": "https://idyllic-market.kr/web/product/big/202607/a898ffb7693ed3298832cb988a5ef4ee.webp"}, {"name": "🍀쓰리피스set #살안타템 #레이어드 뷔스티에 블라우스 가

In [ ]:
# 1. 평가 실행
res = scorer.evaluate_page(
    body_text=body_text,
    image_text=image_text_combined,
    user_queries=test_user_queries,
    json_ld_str=json_ld_str,
    image_list=image_list_for_eval
)

print("=" * 50)
print("📌 Threshold(통과 기준) 적용 전/후 점수 비교")
print("=" * 50)

# 1. 본문 텍스트 비율 (Threshold: 0.2 이상)
#    - 0.2 미만이면 0점, 0.2 이상이면 비율 그대로(0.2~1.0)
print(f"1. 본문 텍스트 비율 점수 : {res.text_ratio_score}")

# 2. 하이브리드 검색 점수 (Threshold: 0.6 이상)
print(f"2. 하이브리드 검색 raw 점수 : {res.hybrid_search.avg_combined_score} (최종 반영: {res.hybrid_search.final_score})")

# 3. 키워드 스터핑 점수 (Threshold: 0.6 이상, stuffing 아닐 때 pass)
print(f"3. 키워드 스터핑 raw 점수   : {res.keyword_stuffing.raw_score} (최종 반영: {res.keyword_stuffing.final_score})")

# 4. JSON-LD 점수 (Threshold: 0.3 이상)
print(f"4. JSON-LD raw 점수         : {res.json_ld.raw_score} (최종 반영: {res.json_ld.final_score})")

# 5. 이미지 Alt 점수 (Threshold: 0.2 이상)
print(f"5. 이미지 Alt raw 점수      : {res.image_alt.raw_avg_score} (최종 반영: {res.image_alt.avg_score})")

print("=" * 50)
print(f"🏆 최종 가공 점수: {res.formatted_score}")


================ [JSON-LD 파싱 완료 내용] ================
{
  "@context": "https://schema.org",
  "@type": "Product",
  "name": "🍀쓰리피스set #살안타템 #레이어드 뷔스티에 블라우스 가디건 세트(2col) 모네블라우스세트",
  "image": [
    "https://idyllic-market.kr/web/product/big/202607/a898ffb7693ed3298832cb988a5ef4ee.webp",
    "https://idyllic-market.kr/web/product/extra/big/202607/37b7acc5eeae92a8d2ee17f3038cc0a8.jpg",
    "https://idyllic-market.kr/web/product/extra/big/202607/505d2fd5211f74912a53c3b999677af2.jpg",
    "https://idyllic-market.kr/web/product/extra/big/202607/48ab86c5b9d861a1f58b3a6230270e3c.jpg"
  ],
  "description": "",
  "brand": {
    "@type": "Brand",
    "name": "아이딜릭"
  },
  "offers": [
    {
      "name": "🍀쓰리피스set #살안타템 #레이어드 뷔스티에 블라우스 가디건 세트(2col) 모네블라우스세트 핑크-F",
      "price": 32900,
      "priceCurrency": "KRW",
      "availability": "InStock",
      "url": "https://idyllic-market.kr/product/🍀쓰리피스set-살안타템-레이어드-뷔스티에-블라우스-가디건-세트2col-모네블라우스세트/3031/?item_code=P0000EMP000A",
      "image": "https://

In [5]:
# Cell 1
import json
import os
from pathlib import Path
from sqlalchemy import create_engine, text

from premodel2_ver210 import GEOScorer

# 스코어러 객체 생성 (전역 모델 로딩)
scorer = GEOScorer()
print("✅ GEOScorer 모델 로드 완료!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22063.03it/s]


✅ GEOScorer 모델 로드 완료!


In [6]:
import json
import os
from pathlib import Path
from sqlalchemy import create_engine, text

def fetch_data_from_db(page_id: int):
    # 1. 환경변수 파일(.env) 로드
    CURRENT_DIR = Path.cwd()
    ENV_PATH = CURRENT_DIR / "../project_db/geo.env"

    if ENV_PATH.exists():
        with open(ENV_PATH, "r", encoding="utf-8-sig") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if "=" in line:
                    key, val = line.split("=", 1)
                    os.environ[key.strip()] = val.strip()

    # 2. DB 연결 정보 추출
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    DB_HOST = os.getenv("DB_HOST")
    DB_PORT = os.getenv("DB_PORT")
    DB_NAME = os.getenv("DB_NAME")

    ENGINE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(ENGINE_URL)

    # 3. DB 데이터 조회
    with engine.connect() as conn:
        # (1) raw_data_table (PK: page_id)
        product_query = text("""
            SELECT 
                product_name, 
                product_cat, 
                text_contents, 
                image_text, 
                json_ld_contents 
            FROM raw_data_table 
            WHERE page_id = :page_id
        """)
        raw_product = conn.execute(product_query, {"page_id": page_id}).mappings().fetchone()

        if not raw_product:
            raise ValueError(f"page_id가 {page_id}인 상품을 raw_data_table에서 찾을 수 없습니다.")

        raw_product = dict(raw_product)

        # json_ld_contents 파싱
        json_ld_data = raw_product['json_ld_contents']
        if isinstance(json_ld_data, str) and json_ld_data.strip():
            try:
                json_ld_data = json.loads(json_ld_data)
            except Exception:
                pass

        # (2) question_table (PK: query_id)
        questions_query = text("""
            SELECT 
                query_id,
                query_text, 
                query_cat, 
                query_keyword 
            FROM question_table
        """)
        raw_questions = conn.execute(questions_query).mappings().fetchall()

        user_queries = []
        for q in raw_questions:
            keywords = q['query_keyword']
            if isinstance(keywords, str):
                try:
                    keywords = json.loads(keywords)
                except Exception:
                    keywords = [x.strip() for x in keywords.split(',') if x.strip()]
            elif keywords is None:
                keywords = []

            user_queries.append({
                "query_text": q['query_text'],
                "category": q['query_cat'],
                "must_have": keywords
            })

        # (3) image_data_table (FK: page_id / PK: page_id, image_sequence)
        image_query = text("""
            SELECT 
                alt_contents 
            FROM image_data_table 
            WHERE page_id = :page_id
            ORDER BY image_sequence ASC
        """)
        raw_images = conn.execute(image_query, {"page_id": page_id}).mappings().fetchall()

        image_list = [
            {
                "alt": img['alt_contents'] or "",
                "is_text_image": False
            }
            for img in raw_images
        ]

        return raw_product, user_queries, json_ld_data, image_list

In [7]:
# 진단할 대상의 page_id 입력
TARGET_PAGE_ID = 101

# DB 추출
raw_product, user_queries, json_ld, image_list = fetch_data_from_db(TARGET_PAGE_ID)

# 스코어러 실행
result = scorer.evaluate_page(
    body_text=raw_product['text_contents'] or "",
    image_text=raw_product['image_text'] or "",
    user_queries=user_queries,
    product_cat=raw_product['product_cat'],
    product_name=raw_product['product_name'],
    json_ld_str=json_ld,
    image_list=image_list
)

In [8]:
result

GEOTotalEvaluationResult(total_score=43.0, text_ratio_score=0.0, hybrid_search=HybridSearchResult(final_score=0.2834, avg_combined_score=0.2834, avg_lexical_overlap=0.2469, avg_cosine_sim_raw=0.3199, total_queries_count=1220, cat_filtered_count=293, must_have_filtered_count=33, query_details=[{'query': '엉덩이를 살짝 덮는 기장의 하프 코트 스타일 자켓 추천해 줘.', 'category': '아우터', 'must_have': ['코트', '하프'], 'lexical_overlap': 0.1667, 'cosine_sim_raw': 0.302, 'query_hybrid_score': 0.2344}, {'query': '힙하고 스트릿한 무드 뿜뿜하는 오버핏 카고 점프수트 보여줘.', 'category': '아우터', 'must_have': ['코트', '점프수트'], 'lexical_overlap': 0.0, 'cosine_sim_raw': 0.2556, 'query_hybrid_score': 0.1278}, {'query': '울 함유량이 높아서 가볍고 따뜻한 여성용 핸드메이드 코트 찾고 있어요.', 'category': '아우터', 'must_have': ['코트', '핸드메이드'], 'lexical_overlap': 0.2, 'cosine_sim_raw': 0.2992, 'query_hybrid_score': 0.2496}, {'query': '출근할 때 슬랙스나 정장에 받쳐 입기 좋은 포멀한 남성 코트 보여주세요.', 'category': '아우터', 'must_have': ['코트'], 'lexical_overlap': 0.125, 'cosine_sim_raw': 0.3294, 'query_hybrid_score': 0.